In [8]:
import os
import re
import sys

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

ROOT_PATH = r'D:\Users\wangy\PycharmProjects\GlassDoor'

if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

from Constants import Constants as const

In [3]:
gd_monthly_df = pd.read_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_monthly_compustat_ann_2008_2023.pkl'))

In [4]:
gd_monthly_df.keys()

Index(['gvkey', 'year', 'month', 'num_reviews', 'rating_mean',
       'career_opp_mean', 'comp_benefits_mean', 'management_mean',
       'work_life_mean', 'culture_values_mean', 'diversity_mean',
       'GD_CompanyName', 'GD_CompanyID', 'at', 'ceq', 'csho', 'dlc', 'dltt',
       'intan', 'pi', 'pifo', 'ppent', 'seq', 'spi', 'tlcf', 'txdfed', 'txdfo',
       'txdi', 'txpd', 'txr', 'txt', 'xrd', 'xsga', 'sic', 'state', 'ipodate',
       'conml'],
      dtype='object')

In [4]:
gd_ann_df = gd_monthly_df.groupby(['gvkey', 'year']).agg({
    'num_reviews': 'count', 'career_opp_mean': 'mean', 'comp_benefits_mean': 'mean', 'management_mean': 'mean',
    'rating_mean': 'mean', 'work_life_mean': 'mean', 'culture_values_mean': 'mean', 'diversity_mean': 'mean',
    'GD_CompanyName': 'first', 'GD_CompanyID': 'first'
    }).reset_index()

In [5]:
# Compute weighted annual averages for Glassdoor metrics
metrics = [
    'rating_mean',
    'career_opp_mean',
    'comp_benefits_mean',
    'management_mean',
    'work_life_mean',
    'culture_values_mean',
    'diversity_mean'
]

# Weighted totals per month
for col in metrics:
    gd_monthly_df[f'{col}_wt_total'] = gd_monthly_df[col] * gd_monthly_df['num_reviews']

# Aggregate to annual at gvkey-year level
agg_dict = {'num_reviews': 'sum', 'GD_CompanyName': 'first', 'GD_CompanyID': 'first'}
for col in metrics:
    agg_dict[f'{col}_wt_total'] = 'sum'

gd_ann_wt_df = gd_monthly_df.groupby(['gvkey', 'year']).agg(agg_dict).reset_index()

# Compute weighted averages; handle zero-review years
_den = gd_ann_wt_df['num_reviews'].replace(0, np.nan)
for col in metrics:
    gd_ann_wt_df[col] = gd_ann_wt_df[f'{col}_wt_total'] / _den
    gd_ann_wt_df.drop(columns=f'{col}_wt_total', inplace=True)

In [8]:
gd_ann_df.describe()

,gvkey,year,num_reviews,career_opp_mean,comp_benefits_mean,management_mean,rating_mean,work_life_mean,culture_values_mean,diversity_mean,GD_CompanyID
count,26748.000000,26748.000000,26748.000000,25903.000000,25653.000000,26031.000000,26748.000000,25823.000000,20666.000000,5302.000000,2.674800e+04
mean,59132.033760,2015.448744,7.117841,2.986580,3.285156,2.880023,3.239274,3.324475,3.203021,3.599401,1.772627e+05
std,64921.874347,4.117687,4.356761,0.763764,0.722401,0.824573,0.762293,0.767743,0.819674,0.713624,4.684169e+05
min,1004.000000,2008.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000e+00
25%,10894.000000,2012.000000,3.000000,2.571429,2.938571,2.432769,2.851668,2.950000,2.759259,3.276239,1.697000e+03
50%,26906.000000,2016.000000,8.000000,3.000000,3.322018,2.932102,3.272727,3.333333,3.241235,3.685693,6.135000e+03
75%,108797.000000,2019.000000,12.000000,3.446905,3.750000,3.333333,3.722222,3.812500,3.715504,4.000000,1.579100e+04
max,351038.000000,2023.000000,12.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,8.131118e+06


In [7]:
gd_ann_wt_df.describe()

,gvkey,year,num_reviews,GD_CompanyID,rating_mean,career_opp_mean,comp_benefits_mean,management_mean,work_life_mean,culture_values_mean,diversity_mean
count,26748.000000,26748.000000,26748.000000,2.674800e+04,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000
mean,59132.033760,2015.448744,141.755533,1.772627e+05,3.246518,2.760671,2.967016,2.682577,3.035234,2.349330,0.561175
std,64921.874347,4.117687,813.901197,4.684169e+05,0.760414,0.941828,1.003587,0.946269,0.992414,1.489763,1.245529
min,1004.000000,2008.000000,1.000000,4.000000e+00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10894.000000,2012.000000,3.000000,1.697000e+03,2.857143,2.333333,2.550000,2.200000,2.625000,1.000000,0.000000
50%,26906.000000,2016.000000,13.000000,6.135000e+03,3.283333,2.916667,3.100000,2.778264,3.153958,2.823431,0.000000
75%,108797.000000,2019.000000,60.000000,1.579100e+04,3.732129,3.333333,3.600000,3.213787,3.636364,3.467317,0.000000
max,351038.000000,2023.000000,57350.000000,8.131118e+06,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [6]:
gd_ann_wt_df.to_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_ann_2008_2023.pkl'))

In [2]:
# Load major customer file
major_corp_cus_df = pd.read_stata(os.path.join(const.MAJ_DATA_PATH, 'MajorCorpCusMeaures_0624.dta'))
major_cus = pd.read_stata(os.path.join(const.MAJ_DATA_PATH, 'MajorCusMeaures_0624.dta'))

In [5]:
major_cus_df = major_corp_cus_df.merge(
    major_cus, on=['gvkey', 'fyear'], how='outer', suffixes=('_chen2025', '_cg2017')
)

In [9]:
major_cus_df.rename(columns={'fyear': 'year'}, inplace=True)

In [13]:
gd_ann_wt_df = pd.read_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_ann_2008_2023.pkl'))
maj_with_gd_df = major_cus_df.merge(
    gd_ann_wt_df, on=['gvkey', 'year'], how='left'
)
gd_ann_wt_df[const.YEAR] -= 1
maj_with_gd_lag_df = maj_with_gd_df.merge(
    gd_ann_wt_df, on=['gvkey', 'year'], how='left', suffixes=('', '_l1')
)
maj_with_gd_lag_df.to_stata(os.path.join(
    const.MAJ_OUTPUT_PATH, 'MajorCustomer_Glassdoor_2008_2023_0110.dta'), write_index=False, version=119)

In [4]:
import wrds
# Connect to WRDS
import os
db = wrds.Connection(wrds_username='wangyouan')
print('WRDS connected:', db is not None)

Loading library list...
Done
WRDS connected: True


In [5]:
# Query Compustat annual fundamentals (North America)
comp_query = """
SELECT gvkey, fyear, datadate, at, lt, che, ceq, txditc, csho, prcc_f, mkvalt, sale, xrd, capx, ppent, ppegt, oibdp, ib, sich, dvpsx_f
FROM comp.funda
WHERE indfmt = 'INDL'
  AND datafmt = 'STD'
  AND consol  = 'C'
  AND popsrc  = 'D'
  AND fyear BETWEEN 2006 AND 2023
"""
funda = db.raw_sql(comp_query)
funda.rename(columns={'fyear': 'year'}, inplace=True)
print('Compustat rows:', len(funda))

# Basic cleaning
funda = funda.drop_duplicates(subset=['gvkey','year'])
funda = funda.sort_values(['gvkey','year'])


Compustat rows: 206597


In [6]:
# Compute common firm control variables
import numpy as np
ctrl = funda.copy()

# Guard against non-positive assets
ctrl.loc[ctrl['at'] <= 0, 'at'] = np.nan

# Market capitalization
ctrl['mktcap'] = ctrl['prcc_f'] * ctrl['csho']

# Size: ln(total assets)
ctrl['size_ln_at'] = np.log(ctrl['at'])

# Leverage: total liabilities / total assets
ctrl['leverage'] = ctrl['lt'] / ctrl['at']

# Tobin's Q (approx): (MktCap + LT - CHE) / AT
ctrl['tobinq_m'] = (ctrl['mktcap'] + ctrl['lt'] - ctrl['che']) / ctrl['at']

# Alternative using MKVALT when available
ctrl['tobinq_alt'] = (ctrl['mkvalt'] + ctrl['lt'] - ctrl['che']) / ctrl['at']
ctrl['tobinq'] = ctrl['tobinq_m'].fillna(ctrl['tobinq_alt'])

# Cash holdings: CHE / AT
ctrl['cash_at'] = ctrl['che'] / ctrl['at']

# ROA: IB / AT (simple version)
ctrl['roa'] = ctrl['ib'] / ctrl['at']

# R&D intensity: XRD / AT
ctrl['rd_at'] = ctrl['xrd'] / ctrl['at']
ctrl['rd_indicator'] = (ctrl['xrd'] > 0).astype('float')

# CapEx intensity: CAPX / AT
ctrl['capx_at'] = ctrl['capx'] / ctrl['at']

# Asset tangibility: PPENT/AT (fallback to PPEGT)
ctrl['tangibility'] = ctrl['ppent'] / ctrl['at']
ctrl.loc[ctrl['tangibility'].isna(), 'tangibility'] = ctrl['ppegt'] / ctrl['at']

# Keep relevant columns (use sich instead of sic and include dvpsx_f)
ctrl_vars = ctrl[['gvkey','year','datadate','sich','at','lt','che','ib','sale','xrd','capx','ppent','ppegt','csho','prcc_f','mkvalt','mktcap','dvpsx_f',
                   'size_ln_at','leverage','tobinq','cash_at','roa','rd_at','rd_indicator','capx_at','tangibility']]
print('Control vars shape:', ctrl_vars.shape)


Control vars shape: (206586, 27)


d:\anaconda3\envs\GlassDoor\Lib\site-packages\pandas\core\arrays\masked.py:672: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


In [9]:
# Compute annual stock return from Compustat prcc_f incl. dividends (dvpsx_f)
ctrl_vars = ctrl_vars.sort_values(['gvkey','year']).copy()
ctrl_vars['prcc_f_lag'] = ctrl_vars.groupby('gvkey')['prcc_f'].shift(1)
ctrl_vars['year_diff'] = ctrl_vars.groupby('gvkey')['year'].diff()
ctrl_vars.loc[ctrl_vars['year_diff'] != 1, 'prcc_f_lag'] = np.nan
ctrl_vars['dvpsx_f'] = pd.to_numeric(ctrl_vars['dvpsx_f'], errors='coerce').fillna(0)
ctrl_vars['stock_return'] = (ctrl_vars['prcc_f'] + ctrl_vars['dvpsx_f']) / ctrl_vars['prcc_f_lag'] - 1
print('Computed stock_return rows:', ctrl_vars['stock_return'].notna().sum())

Computed stock_return rows: 164013


In [10]:
ctrl_vars.describe()

,year,sich,at,lt,che,ib,sale,xrd,capx,ppent,...,tobinq,cash_at,roa,rd_at,rd_indicator,capx_at,tangibility,prcc_f_lag,year_diff,stock_return
count,206586.0,129851.0,154235.0,154668.0,154891.0,154130.0,154124.0,69792.0,152945.0,151197.0,...,138058.0,154210.0,153462.0,69604.0,69792.000000,152294.0,150518.0,164366.0,181294.0,164013.0
mean,2014.725364,4804.922704,13461.157032,11085.249559,1569.316812,231.900599,3404.159047,159.262377,248.615471,1697.056968,...,56.51928,0.223746,-3.457269,0.417947,0.792311,0.053118,0.251817,34.519776,1.000469,3.750247
std,5.200608,2104.315277,111750.079992,104421.75537,18688.307751,1771.735001,16535.535989,1051.541375,1411.610337,8980.105884,...,2575.35419,0.2695,359.06495,12.456736,0.405655,0.47642,0.288314,624.535525,0.033459,467.93975
min,2006.0,100.0,0.001,-0.103,-8.03,-99289.0,-24954.684,-18.542,-3258.0,0.0,...,-0.990717,-1.179775,-130077.0,-28.444444,0.000000,-2.771739,0.0,0.000001,1.0,-1.0
25%,2010.0,2870.0,39.845,11.88075,3.879,-8.049,8.78175,0.213,0.204,2.045,...,0.908195,0.031611,-0.159609,0.003075,1.000000,0.002207,0.021208,3.123225,1.0,-0.226377
50%,2015.0,4899.0,375.433,165.274,32.746,1.293,133.0955,6.48,4.2,27.736,...,1.229906,0.103402,0.005694,0.041522,1.000000,0.018,0.117728,14.16,1.0,0.026946
75%,2019.0,6331.0,2428.1945,1503.3645,184.0675,58.036,1109.5925,41.09625,50.841,329.5,...,2.210914,0.315336,0.047084,0.163546,1.000000,0.053412,0.422321,31.23,1.0,0.260143
max,2023.0,9998.0,4325437.0,4247755.0,1058786.122,104821.0,645737.0,85622.0,63645.0,292684.091,...,650188.25,1.0,13583.0,1981.5,1.000000,79.714286,1.455621,141600.0,6.0,163493.0


In [12]:
# Save firm controls (winsorized 1%-99%, infinities -> NaN)
import numpy as np

# Select final columns (uses sich and Compustat-based stock_return)
final_cols = ['gvkey','year','sich',
              'size_ln_at','leverage','tobinq','cash_at','roa','rd_at','rd_indicator','capx_at','tangibility','stock_return']
firm_controls = ctrl_vars[final_cols].copy()

# Replace infinities with NaN
firm_controls.replace([np.inf, -np.inf], np.nan, inplace=True)

# Winsorize numeric control variables at 1% and 99%
num_cols = ['size_ln_at','leverage','tobinq','cash_at','roa','rd_at','capx_at','tangibility','stock_return']
for col in num_cols:
    q_low = firm_controls[col].quantile(0.01)
    q_high = firm_controls[col].quantile(0.99)
    firm_controls[col] = firm_controls[col].clip(lower=q_low, upper=q_high)

# Persist to TEMP and optional Stata
temp_pkl = os.path.join(const.MAJ_TEMP_PATH, 'firm_controls_2008_2023.pkl')
firm_controls.to_pickle(temp_pkl)
print('Saved:', temp_pkl)


Saved: D:\Onedrive\Temp\Projects\MajorCustomer\temp\firm_controls_2008_2023.pkl


In [15]:
firm_controls.replace(pd.NA, np.nan, inplace=True)
base_reg_df = pd.read_stata(os.path.join(
    const.MAJ_OUTPUT_PATH, 'MajorCustomer_Glassdoor_2008_2023_0110.dta'))
firm_controls[const.GVKEY] = firm_controls['gvkey'].astype(int)
reg_with_ctrl_df = base_reg_df.merge(firm_controls, on=['gvkey', 'year'], how='left')
reg_with_ctrl_df.to_stata(os.path.join(const.MAJ_OUTPUT_PATH, '20260110_maj_customer_gd_with_ctrl.dta'),
                          write_index=False, version=119)

In [17]:
base_reg_df.shape

(57908, 32)

In [16]:
reg_with_ctrl_df.shape

(57908, 43)